# A systolic array, cycle by cycle
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week1/Systolic_Array_Cycles_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 1, Module 2: Where scheduling decisions live**  
**Author:** Dr. Kaoutar El Maghraoui

A systolic array is a grid of simple multiply-accumulate (MAC) cells. Each cell talks only to its neighbors, and values are reused as they move through the grid instead of being fetched from memory for every multiply. The lecture animates a 4 by 4 weight-stationary array: weights stay in the cells, activations enter from the left and move right, and partial sums move down.

In this notebook you build that array in NumPy and run it one clock cycle at a time. You check that it computes the same result as `A @ W`, watch the wave of active cells fill and drain the array, and measure how much of the array does useful work for small and large inputs. Everything runs on the CPU.

**What you will do**
1. Run one MAC cell with the numbers from the lecture.
2. Build a weight-stationary array and check its result against NumPy.
3. Print the active cells at every cycle for a 4 by 4 example.
4. Measure utilization as the input grows, and relate it to the TPU v1 array.
5. Compare with an output-stationary mapping of the same matmul.

**Before you start:** any runtime works, including CPU. The notebook takes under a minute.

## 1. One MAC cell
A cell holds one weight. Each cycle it receives an activation from the left and a partial sum from above. It multiplies the activation by its weight, adds the product to the partial sum, passes the activation to the right and passes the new partial sum down.

The lecture example: weight 2, activation 3, incoming partial sum 10.

In [ ]:
import numpy as np


def mac_cell(weight, activation_in, psum_in):
    """One weight-stationary cell: returns (activation_out, psum_out)."""
    psum_out = psum_in + activation_in * weight
    return activation_in, psum_out


a_out, p_out = mac_cell(weight=2, activation_in=3, psum_in=10)
print(f"activation passed right: {a_out}")
print(f"partial sum passed down: {p_out}   (10 + 3 x 2)")

The cell does not know which row or column of the matrix it is working on. The schedule, meaning which value arrives at which cycle, makes the result correct. In a real accelerator that schedule is fixed by the hardware and the compiler.

## 2. A weight-stationary array
The array computes `C = A @ W`, where `A` is M by K and `W` is K by N. Cell `(k, n)` holds `W[k, n]` for the whole run.

The timing rules:
- **Activations are skewed.** Row `m` of `A` enters array row `k` at cycle `m + k`, then moves one cell to the right per cycle.
- **Partial sums move down** one cell per cycle, starting from zero at the top row.
- **Results leave the bottom.** Column `n` emits `C[m, n]` at cycle `m + (K - 1) + n`.

The skew makes the value from row `m` reach cell `(k, n)` at the same cycle as the partial sum for row `m`, so every product is added to the right sum. The whole run takes `M + K + N - 2` cycles.

In [ ]:
def systolic_matmul(A, W, record=False):
    """Simulate a weight-stationary systolic array computing A @ W.

    Returns the result, the number of active cells at each cycle, and,
    if record is True, a boolean grid of active cells for each cycle.
    """
    M, K = A.shape
    K2, N = W.shape
    assert K == K2, "inner dimensions must match"

    act = np.zeros((K, N))            # activation held in each cell this cycle
    psum = np.zeros((K, N))           # partial sum leaving each cell downward
    valid = np.zeros((K, N), bool)    # which cells hold a real activation
    C = np.zeros((M, N))
    active, frames = [], []

    for t in range(M + K + N - 2):
        # Activations move one cell to the right.
        new_act = np.zeros((K, N))
        new_valid = np.zeros((K, N), bool)
        new_act[:, 1:] = act[:, :-1]
        new_valid[:, 1:] = valid[:, :-1]

        # The left edge injects the skewed input: array row k gets A[t - k, k].
        for k in range(K):
            m = t - k
            if 0 <= m < M:
                new_act[k, 0] = A[m, k]
                new_valid[k, 0] = True

        # Partial sums arrive from the cell above (zero at the top row),
        # then every cell does one multiply-accumulate with its own weight.
        psum_in = np.zeros((K, N))
        psum_in[1:, :] = psum[:-1, :]
        new_psum = psum_in + new_act * W

        # Finished sums leave the bottom row.
        for n in range(N):
            m = t - (K - 1) - n
            if 0 <= m < M:
                C[m, n] = new_psum[K - 1, n]

        act, psum, valid = new_act, new_psum, new_valid
        active.append(int(valid.sum()))
        if record:
            frames.append(valid.copy())
    return C, active, frames


rng = np.random.default_rng(0)
A = rng.integers(-3, 4, size=(4, 4)).astype(float)
W = rng.integers(-3, 4, size=(4, 4)).astype(float)

C, active, frames = systolic_matmul(A, W, record=True)
print("A @ W from NumPy:\n", A @ W)
print("\nResult from the systolic array:\n", C)
print("\nIdentical:", np.array_equal(C, A @ W))
print("Cycles:", len(active), " (M + K + N - 2 =", 4 + 4 + 4 - 2, ")")

## 3. Watch the wave
Each grid below is the array at one cycle. `#` marks a cell doing a useful multiply-accumulate, `.` marks an idle cell. The weights never move. Only activations and partial sums do.

In [ ]:
for t, grid in enumerate(frames):
    rows = ["".join("#" if cell else "." for cell in row) for row in grid]
    print(f"cycle {t:2d}  active {active[t]:2d}/16   " + "  ".join(rows))

**Reading the output**
- The first cycles are the **fill**: only cells near the top-left corner have valid inputs.
- Activity peaks at 12 of 16 cells at cycles 4 and 5. With only 4 input rows, the array never has all 16 cells busy at once. The steady state that the peak throughput figure assumes needs many more rows.
- The last cycles are the **drain**: the last rows of `A` move through the bottom-right corner.
- 64 useful MACs took 10 cycles on 16 cells, so the array was busy for 64 / 160 = 40 percent of its cell-cycles. One MAC unit would need 64 cycles. The exact ratio depends on how fill and drain are counted, which is why the lecture says not to focus on the exact number.

## 4. Utilization grows with the input
Fill and drain cost about `K + N` cycles, once per matrix. The useful work grows with `M`, the number of input rows streamed through. Utilization is the useful MACs divided by the cell-cycles:

`utilization = (M x K x N) / (K x N x (M + K + N - 2)) = M / (M + K + N - 2)`

The cell below checks the formula against the simulator for a 32 by 32 array, then applies it to the 256 by 256 array of the first-generation TPU.

In [ ]:
K = N = 32
print(f"Array {K} x {N}")
print(f"{'M rows':>8} {'cycles':>8} {'simulated':>10} {'formula':>9}")
for M in (4, 32, 128, 256):
    A = rng.standard_normal((M, K))
    W = rng.standard_normal((K, N))
    C, active, _ = systolic_matmul(A, W)
    assert np.allclose(C, A @ W)
    sim = sum(active) / (K * N * len(active))
    print(f"{M:8d} {len(active):8d} {sim:10.1%} {M / (M + K + N - 2):9.1%}")

print("\nTPU v1 matrix unit, 256 x 256 cells (formula only)")
for M in (8, 256, 2048, 16384):
    print(f"  M = {M:6d}: utilization {M / (M + 256 + 256 - 2):6.1%}")

In [ ]:
import matplotlib.pyplot as plt

Ms = np.logspace(0, 5, 200)
fig, ax = plt.subplots(figsize=(6, 3.2))
for size, style in ((32, "-"), (128, "--"), (256, ":")):
    ax.plot(Ms, Ms / (Ms + 2 * size - 2), style, color="black", label=f"{size} x {size} array")
ax.set_xscale("log")
ax.set_xlabel("rows of A streamed through the array (M)")
ax.set_ylabel("utilization")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.set_title("Fill and drain matter for small inputs")
plt.tight_layout()
plt.show()

**Reading the output**
- The simulator and the formula agree.
- A 256 by 256 array fed 8 rows is busy about 1.5 percent of the time. It needs thousands of rows to approach its peak.
- This is why small matrix operations reach a small fraction of a systolic unit's headline throughput, and why compilers try to build large, regular tiles. During LLM decode at batch size 1, `M` is 1.

## 5. Output-stationary: a different choice of what stays still
The lecture also shows an output-stationary array. Cell `(m, n)` keeps a local accumulator for `C[m, n]`. Values of `A` move right, values of `W` move down, and both are skewed so that `A[m, k]` and `W[k, n]` meet at cell `(m, n)` at cycle `m + n + k`.

Both designs are systolic, because communication is local and regular. They differ in which tensor stays in place, which changes where reuse happens and which movements are expensive.

In [ ]:
def output_stationary_matmul(A, W):
    """Simulate an output-stationary array: cell (m, n) accumulates C[m, n]."""
    M, K = A.shape
    _, N = W.shape
    acc = np.zeros((M, N))
    cycles = M + N + K - 2
    for t in range(cycles):
        for m in range(M):
            for n in range(N):
                k = t - m - n          # the A and W values that meet here at cycle t
                if 0 <= k < K:
                    acc[m, n] += A[m, k] * W[k, n]
    return acc, cycles


A = rng.standard_normal((4, 4))
W = rng.standard_normal((4, 4))
C_os, cycles_os = output_stationary_matmul(A, W)
C_ws, active_ws, _ = systolic_matmul(A, W)
print("Output-stationary matches NumPy:", np.allclose(C_os, A @ W), "| cycles:", cycles_os)
print("Weight-stationary matches NumPy:", np.allclose(C_ws, A @ W), "| cycles:", len(active_ws))

**Reading the output**
- Both mappings compute the same matrix in the same number of cycles for this square example.
- What changes is the data movement. Weight-stationary loads each weight once and streams activations and partial sums. Output-stationary streams both inputs and never moves the partial sums until the end. When someone says an accelerator uses a systolic array, ask which dataflow is mapped onto it.

## Try this
1. In section 2, use a 4 by 8 `A` and an 8 by 4 `W`. Predict the cycle count from `M + K + N - 2` before you run it.
2. In section 4, set `K = N = 8` and compare the utilization with the 32 by 32 array for the same `M`.
3. Change `mac_cell` so it ignores the incoming partial sum. Run section 2 and see which entries of `C` are wrong.

## Summary
- A systolic cell does one multiply-accumulate and talks only to its neighbors. The schedule makes the result correct.
- A weight-stationary array streams activations and partial sums past fixed weights. An output-stationary array keeps the partial sums in place.
- Fill and drain take about `K + N` cycles per matrix, so utilization is `M / (M + K + N - 2)`. Small inputs use a small fraction of a large array.
- The arithmetic is the same as `A @ W`. What changes is who organizes the work and when.

Lecture: Week 1, Module 2, slides 25 to 31 (systolic array, one cell, TPU v1 matrix unit, systolic wave).